# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imtiyazsoomro/flyrank-ml-internship-imtiyaz/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [8]:
import os
import pandas as pd
import numpy as np

# 1. Setup environment and load data
if not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    !git clone https://github.com/imtiyazsoomro/flyrank-ml-internship-imtiyaz.git
    %cd flyrank-ml-internship-imtiyaz

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# 2. Define target (y)
y = (df['trend_direction'] == 'down').astype(int)

# 3. Define and build feature vector (X)
feature_cols = [
    'content_age_days',
    'days_since_last_update',
    'impressions_90d',
    'avg_position',
    'word_count'
]
X = df[feature_cols].copy()

# 4. Handle any missing values (impute with median to be safe)
X = X.fillna(X.median())

print(f"Feature vector X successfully built with shape: {X.shape}")
print("-" * 40)
print(X.head(3))

Feature vector X successfully built with shape: (30000, 5)
----------------------------------------
   content_age_days  days_since_last_update  impressions_90d  avg_position  \
0               187                      20             3803          10.6   
1               445                      25            15320          20.3   
2               141                      20            12581          36.5   

   word_count  
0      3221.0  
1      2481.0  
2      3515.0  


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [9]:
# Generate metadata for the feature vector
feature_notes = pd.DataFrame({
    'Feature': X.columns,
    'Missing_Values_Remaining': X.isnull().sum().values,
    'Data_Type': X.dtypes.values
})

print("--- Feature Vector Metadata Audit ---")
print(feature_notes.to_string(index=False))

--- Feature Vector Metadata Audit ---
               Feature  Missing_Values_Remaining Data_Type
      content_age_days                         0     int64
days_since_last_update                         0     int64
       impressions_90d                         0     int64
          avg_position                         0   float64
            word_count                         0   float64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [10]:
# Attack the features: Check Pearson correlation against the target label 'y'
correlations = X.apply(lambda col: col.corr(y))

print("--- Leakage Hunt: Feature Correlation with Target (y) ---")
print(correlations.sort_values(ascending=False))
print("\nConclusion: All correlations are relatively weak to moderate. No single feature perfectly predicts the label, confirming no obvious mechanical leakage in X.")

--- Leakage Hunt: Feature Correlation with Target (y) ---
word_count                0.084279
days_since_last_update    0.081383
impressions_90d          -0.018175
avg_position             -0.029035
content_age_days         -0.163882
dtype: float64

Conclusion: All correlations are relatively weak to moderate. No single feature perfectly predicts the label, confirming no obvious mechanical leakage in X.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [11]:
# Explicitly list and verify excluded columns
excluded_cols = ['trend_direction', 'trend_pct']

# Verify none of these made it into X
leaked_into_X = [col for col in excluded_cols if col in X.columns]

if not leaked_into_X:
    print(f"Success: Excluded columns {excluded_cols} are safely kept out of the feature vector.")
else:
    print(f"WARNING: Leakage detected! {leaked_into_X} found in X.")

Success: Excluded columns ['trend_direction', 'trend_pct'] are safely kept out of the feature vector.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.